# Apex workshop config

Set catalog, schema, and SQL warehouse once. Later notebooks import `shared.config.load_config` and fail fast if these are still placeholders.


## Widgets

Use your own Unity Catalog names. Do not commit a real workspace hostname, warehouse id, or email.


In [ ]:
dbutils.widgets.text("catalog", "YOUR_CATALOG", "Catalog")
dbutils.widgets.text("schema", "YOUR_SCHEMA", "Schema")
dbutils.widgets.text("warehouse_id", "YOUR_WAREHOUSE_ID", "SQL warehouse id")
dbutils.widgets.text("repo_root", "", "Workspace folder that contains shared/")


## Load and validate

Adds the repo `shared/` folder to sys.path, then validates identifiers so a missing catalog fails before any table is created.


In [ ]:
import sys
from pathlib import Path

root = None
repo_root_w = dbutils.widgets.get("repo_root").strip()
if repo_root_w and (Path(repo_root_w) / "shared" / "config.py").exists():
    root = Path(repo_root_w)
if root is None:
    for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        if (candidate / "shared" / "config.py").exists():
            root = candidate
            break
if root is None:
    raise FileNotFoundError("Set repo_root to the workspace folder that contains shared/")
sys.path.insert(0, str(root))

from shared.config import load_config

cfg = load_config()
print("catalog", cfg.catalog)
print("schema", cfg.schema)
print("warehouse_id set:", bool(cfg.warehouse_id))
print("volume_path", cfg.volume_path)
